# Isolated TFLite Conversion

This notebook builds a separate float32 TFLite export without touching the current conversion flow.

It performs these steps:
- checks conversion prerequisites and fails early if `onnx2tf` is missing
- rebuilds the latest trained MobileNetV3 checkpoint on CPU
- exports an isolated ONNX model into `models/_tflite_build/`
- converts that ONNX model to float32 TFLite with `onnx2tf`
- publishes suffixed `.tflite` and metadata files into `models/` and `mobile/`
- validates TFLite parity against ONNX Runtime on a real test image

## Prerequisites And Imports

In [ ]:
import importlib.util
import json
import shutil
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
from PIL import Image

required_modules = {
    'onnx2tf': 'onnx2tf',
    'tf_keras': 'tf-keras',
    'onnx_graphsurgeon': 'onnx-graphsurgeon',
    'sng4onnx': 'sng4onnx',
    'ai_edge_litert': 'ai-edge-litert',
    'sne4onnx': 'sne4onnx',
}

for module_name, package_name in required_modules.items():
    if importlib.util.find_spec(module_name) is None:
        raise ImportError(
            f"{package_name} is not installed. Install notebook dependencies with: "
            "pip install -r requirements-tflite.txt"
        )

try:
    import tf_keras
    import onnx_graphsurgeon
    import sng4onnx
    import ai_edge_litert
    import sne4onnx
except Exception as exc:
    raise ImportError(
        "The ONNX to TFLite runtime dependencies are not usable. Install notebook dependencies with: "
        "pip install -r requirements-tflite.txt"
    ) from exc

import tensorflow as tf
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import MobileNet_V3_Large_Weights

print(f"Python executable: {sys.executable}")
print(f"TensorFlow: {version('tensorflow')}")
print(f"PyTorch: {torch.__version__}")
print(f"ONNX: {onnx.__version__}")
print(f"ONNX Runtime: {ort.__version__}")
print(f"onnx2tf: {version('onnx2tf')}")
print(f"tf-keras: {version('tf-keras')}")
print(f"onnx-graphsurgeon: {version('onnx-graphsurgeon')}")
print(f"sng4onnx: {version('sng4onnx')}")
print(f"ai-edge-litert: {version('ai-edge-litert')}")
print(f"sne4onnx: {version('sne4onnx')}")

## Load The Latest Trained Checkpoint

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'models').exists() and (cwd / 'mobile').exists():
        return cwd
    if cwd.name == 'notebooks' and (cwd.parent / 'models').exists() and (cwd.parent / 'mobile').exists():
        return cwd.parent
    raise FileNotFoundError('Could not resolve the project root from the current working directory.')


project_root = resolve_project_root()
models_dir = project_root / 'models'
mobile_dir = project_root / 'mobile'
data_dir = project_root / 'data'
build_dir = models_dir / '_tflite_build'
onnx2tf_output_dir = build_dir / 'onnx2tf_output'
build_dir.mkdir(parents=True, exist_ok=True)
onnx2tf_output_dir.mkdir(parents=True, exist_ok=True)

saved_models = sorted(models_dir.glob('plant_disease_mobilenetv3_best_*.pth'))
if not saved_models:
    raise FileNotFoundError('No saved checkpoint found. Run 03_model_training.ipynb first.')

checkpoint_path = saved_models[-1]
checkpoint = torch.load(checkpoint_path, map_location='cpu')
num_classes = checkpoint['num_classes']

model = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
for parameter in model.parameters():
    parameter.requires_grad = False

model.classifier = nn.Sequential(
    nn.Linear(960, 1280),
    nn.Hardswish(),
    nn.Dropout(p=0.3),
    nn.Linear(1280, num_classes),
)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Project root: {project_root}")
print(f"Checkpoint: {checkpoint_path.name}")
print(f"Classes: {num_classes}")
print(f"Build directory: {build_dir}")

## Export An Isolated ONNX Model

In [ ]:
sample_input = torch.randn(1, 3, 224, 224)
onnx_path = build_dir / 'plant_disease_mobilenetv3_tflite.onnx'

torch.onnx.export(
    model,
    sample_input,
    str(onnx_path),
    export_params=True,
    dynamo=False,
    opset_version=13,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

print(f"Saved ONNX: {onnx_path}")
print(f"ONNX size: {onnx_path.stat().st_size / 1024 / 1024:.2f} MB")

## Convert ONNX To Float32 TFLite

In [ ]:
for stale_tflite in onnx2tf_output_dir.rglob('*.tflite'):
    stale_tflite.unlink()

command = [
    sys.executable,
    '-m',
    'onnx2tf',
    '-i',
    str(onnx_path),
    '-o',
    str(onnx2tf_output_dir),
    '-cotof',
]

print('Running conversion command:')
print(' '.join(command))

result = subprocess.run(
    command,
    cwd=project_root,
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    raise RuntimeError('onnx2tf conversion failed. See the command output above.')

print('onnx2tf conversion completed successfully.')

tflite_candidates = sorted(
    path for path in onnx2tf_output_dir.rglob('*.tflite')
    if 'float32' in path.stem.lower()
)
if len(tflite_candidates) != 1:
    raise RuntimeError(
        f'Expected exactly one float32 TFLite file in {onnx2tf_output_dir}, found {len(tflite_candidates)}: '
        f'{[path.name for path in tflite_candidates]}'
    )

generated_tflite_path = tflite_candidates[0]
published_model_path = models_dir / 'plant_disease_mobilenetv3_float32.tflite'
published_mobile_path = mobile_dir / 'plant_disease_mobilenetv3_float32.tflite'

shutil.copy2(generated_tflite_path, published_model_path)
shutil.copy2(generated_tflite_path, published_mobile_path)

print(f"Generated TFLite: {generated_tflite_path}")
print(f"Published to models/: {published_model_path.name}")
print(f"Published to mobile/: {published_mobile_path.name}")
print(f"TFLite size: {published_model_path.stat().st_size / 1024 / 1024:.2f} MB")

## Publish TFLite Metadata

In [ ]:
base_metadata_path = models_dir / 'model_metadata.json'
base_metadata = json.loads(base_metadata_path.read_text(encoding='utf-8'))

tflite_metadata = dict(base_metadata)
tflite_metadata['input_size'] = [1, 224, 224, 3]
tflite_metadata['input_layout'] = 'NHWC'
tflite_metadata['exported_formats'] = dict(base_metadata.get('exported_formats', {}))
tflite_metadata['exported_formats']['tflite_float32'] = published_model_path.name

published_metadata_model_path = models_dir / 'plant_disease_mobilenetv3_float32_metadata.json'
published_metadata_mobile_path = mobile_dir / 'plant_disease_mobilenetv3_float32_metadata.json'

published_metadata_model_path.write_text(json.dumps(tflite_metadata, indent=2), encoding='utf-8')
published_metadata_mobile_path.write_text(json.dumps(tflite_metadata, indent=2), encoding='utf-8')

print(f"Published metadata: {published_metadata_model_path.name}")
print(f"Published metadata: {published_metadata_mobile_path.name}")

## Validate TFLite Against ONNX Runtime

In [ ]:
test_images = [
    path for path in sorted((data_dir / 'test').rglob('*'))
    if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
]
if not test_images:
    raise FileNotFoundError(f'No test images found under {(data_dir / "test")}')

test_image_path = test_images[0]
mean = np.array(tflite_metadata['normalization']['mean'], dtype=np.float32)
std = np.array(tflite_metadata['normalization']['std'], dtype=np.float32)

image = Image.open(test_image_path).convert('RGB').resize((224, 224))
image_array = np.asarray(image, dtype=np.float32) / 255.0
normalized_nhwc = (image_array - mean) / std

onnx_input = np.transpose(normalized_nhwc, (2, 0, 1))[None, ...].astype(np.float32)
tflite_input = normalized_nhwc[None, ...].astype(np.float32)

ort_session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
ort_output = ort_session.run(None, {ort_session.get_inputs()[0].name: onnx_input})[0]

interpreter = tf.lite.Interpreter(model_path=str(published_model_path))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

if list(input_details['shape']) != [1, 224, 224, 3]:
    raise AssertionError(f"Unexpected TFLite input shape: {input_details['shape']}")
if input_details['dtype'] is not np.float32:
    raise AssertionError(f"Unexpected TFLite input dtype: {input_details['dtype']}")

interpreter.set_tensor(input_details['index'], tflite_input)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_details['index'])

onnx_top1 = int(np.argmax(ort_output[0]))
tflite_top1 = int(np.argmax(tflite_output[0]))
max_abs_diff = float(np.max(np.abs(ort_output - tflite_output)))

if onnx_top1 != tflite_top1:
    raise AssertionError(f'Top-1 mismatch: ONNX={onnx_top1}, TFLite={tflite_top1}')
if max_abs_diff > 1e-2:
    raise AssertionError(f'Max abs diff {max_abs_diff:.6f} exceeds threshold 1e-2')

logits = tflite_output[0]
exp_logits = np.exp(logits - np.max(logits))
probabilities = exp_logits / np.sum(exp_logits)
predicted_index = int(np.argmax(probabilities))

print(f"Test image: {test_image_path.relative_to(project_root)}")
print(f"TFLite input shape: {input_details['shape'].tolist()}")
print(f"Predicted class: {tflite_metadata['class_list'][predicted_index]}")
print(f"Confidence: {probabilities[predicted_index] * 100:.2f}%")
print(f"Max abs diff vs ONNX: {max_abs_diff:.6f}")
print('Validation checks passed.')